# Process results from Referencegame

In [1]:
import os, json, re
from typing import List, Dict, Optional, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/project/train_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RESULTS_PATH = "./results_v2"
GAME = "referencegame"
OUTPUT_PATH = "./referencegame_data_v2.json"
MODEL_VERSION = "llama_v2"  # your behavior id

MODEL_ID_FOR_BEHAVIOR = {
    # need to de-comment the version I want to process the files of :)
    #"llama_v0": "meta-llama/Llama-3.1-8B-Instruct",
    #"llama_v1": "imge/reinforce_llama_v1.0",
    "llama_v2": "imge/rf_llama_merged_v2"
}

In [3]:
def _bf16_supported() -> bool:
    return torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

print("Iterate through folder of played games")

data: List[Dict] = []

for experiment in os.scandir(RESULTS_PATH):
    if not experiment.is_dir():
        continue

    current_path = os.path.join(experiment.path, GAME)
    if not os.path.isdir(current_path):
        continue

    for game_mode in os.scandir(current_path):
        if not game_mode.is_dir():
            continue

        for episode in os.scandir(game_mode.path):
            if not episode.is_dir():
                continue
            if "Imge" in episode.path:
                continue

            score_file_path = os.path.join(episode.path, "scores.json")
            interaction_file_path = os.path.join(episode.path, "interactions.json")
            instance_file_path = os.path.join(episode.path, "instance.json")

            try:
                with open(score_file_path, "r") as f:
                    score_data = json.load(f)
            except Exception:
                # No scoring → skip
                continue

            try:
                with open(interaction_file_path, "r") as f:
                    interaction_data = json.load(f)
            except Exception:
                # No interaction → skip
                continue

            try:
                with open(instance_file_path, "r") as f:
                    instance_data = json.load(f)
            except Exception:
                # No instance → skip
                continue

            # Keep only episodes where human is involved
            if "human" not in score_file_path:
                continue

            dp: Dict = {}
            dp["id"] = len(data) + 1
            dp["source"] = episode.path
            dp["behavior_model_id"] = MODEL_VERSION  # <-- needed for IPS mapping

            # Decide role by path order of substrings
            if score_file_path.find("human") > score_file_path.find("llama"):
                dp["role"] = "gen"
            else:
                dp["role"] = "comp"

            # Pull turns
            try:
                prompt_speaker = interaction_data["turns"][0][0]["action"]["content"]
                utterance = interaction_data["turns"][0][1]["action"]["content"]
            except Exception:
                # malformed episode
                continue

            # Map to the field names used downstream
            dp["prompt_speaker"] = prompt_speaker
            dp["utterance"] = utterance

            aborted_at_p1 = score_data.get("episode scores", {}).get("Aborted at Player 1", 0) == 1
            aborted = score_data.get("episode scores", {}).get("Aborted", 0) == 1
            lose = score_data.get("episode scores", {}).get("Lose", 0) == 1

            # Listener side only exists if not aborted at P1
            if aborted_at_p1 and dp["role"] == "comp":
                continue
            try:
                dp["prompt_listener"] = interaction_data["turns"][0][3]["action"]["content"]
                dp["guess"] = interaction_data["turns"][0][4]["action"]["content"]
            except Exception:
                # If missing, leave them absent; fill_logp_behavior will skip
                pass

            # Target text (be careful about indexing)
            tg = instance_data.get("target_grid_name")
            if isinstance(tg, list) and tg:
                target_name = tg[0]
            else:
                target_name = tg  # string or None
            dp["target_text"] = f"Answer: {target_name}"

            # Reward & status
            if aborted or lose:
                dp["reward"] = -1
                dp["status"] = "aborted" if aborted else "finished"
            else:
                dp["reward"] = 1
                dp["status"] = "finished"

            data.append(dp)

print(f"Collected {len(data)} datapoints")

Iterate through folder of played games
Collected 503 datapoints


In [4]:
# ----------------------------
# Behavior log-prob reconstruction (negatives only)
# ----------------------------

def load_model_and_tok(ckpt: str):
    print(f"Loading behavior model: {ckpt}")
    tok = AutoTokenizer.from_pretrained(ckpt, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if _bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        ckpt,
        torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    return tok, model

@torch.no_grad()
def seq_prob(tok, model, prompt: str, action: str, max_len=4096) -> Dict[str, float]:
    """
    Compute the probability of the response (action) given the prompt,
    as required for IPS weighting (Kojima et al. 2021).
    
    Returns:
        {
            "log_prob_sum": log P(action | prompt),
            "seq_prob": P(action | prompt),  # beware: tiny values
            "log_prob_avg": average per-token log-prob
        }
    """
    if not prompt or not action:
        return {"log_prob_sum": float("nan"),
                "seq_prob": float("nan"),
                "log_prob_avg": float("nan")}
    
    print("Encode prompt and action")
    enc_p = tok(prompt, add_special_tokens=False)
    enc_a = tok(action, add_special_tokens=False)
    
    print("Concatinate prompt and action")
    input_ids = enc_p["input_ids"] + enc_a["input_ids"]
    labels    = [-100]*len(enc_p["input_ids"]) + enc_a["input_ids"]

    print("Remove overflow, what is longer than max_len")
    if len(input_ids) > max_len:
        overflow = len(input_ids) - max_len
        input_ids = input_ids[overflow:]
        labels    = labels[overflow:]

    ii  = torch.tensor([input_ids], dtype=torch.long, device=model.device)
    am  = torch.ones_like(ii)
    lab = torch.tensor([labels], dtype=torch.long, device=model.device)

    out = model(input_ids=ii, attention_mask=am, return_dict=True)
    logprobs = torch.log_softmax(out.logits, dim=-1)

    mask = (lab != -100)
    lab_safe = torch.where(mask, lab, torch.zeros_like(lab))
    tok_lp = torch.gather(logprobs, -1, lab_safe.unsqueeze(-1)).squeeze(-1)

    # summed log-prob of the response sequence
    log_prob_sum = (tok_lp * mask).sum()
    print(f"log_prob_sum: {log_prob_sum}")

    # actual probability (may underflow for long sequences)
    seq_prob = log_prob_sum.exp().item()
    print(f"seq_prob: {seq_prob}")

    # average log-prob per token (length normalized)
    log_prob_avg = log_prob_sum / mask.sum()
    print(f"log_prob_avg: {log_prob_avg}")

    return log_prob_sum.item()

def fill_logp_behavior(rows: List[Dict]) -> List[Dict]:
    print("Reconstruct behavior log-probs for negatives…")
    # bucket by behavior model
    buckets: Dict[str, List[int]] = {}
    for i, r in enumerate(rows):
        if r.get("reward", 0) == -1:
            bid = r.get("behavior_model_id")
            if bid:
                buckets.setdefault(bid, []).append(i)

    for beh_id, idxs in buckets.items():
        ckpt = MODEL_ID_FOR_BEHAVIOR[beh_id]
        tok, model = load_model_and_tok(ckpt)
        for i in idxs:
            r = rows[i]
            if r.get("role") == "gen":
                prompt, action = r.get("prompt_speaker"), r.get("utterance")
            else:  # comp
                prompt, action = r.get("prompt_listener"), r.get("guess")
            # skip if missing (e.g., aborted at P1 for comp)
            if not prompt or not action:
                r["logp_behavior"] = None
                continue
            val = seq_prob(tok, model, prompt, action)
            # Optionally guard megative‑inf/NaN
            if not (val == val) or val == float("-inf") or val == float("inf"):
                r["logp_behavior"] = None
            else:
                r["logp_behavior"] = float(val)
    return rows


In [5]:
data = fill_logp_behavior(data)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Wrote {len(data)} rows → {OUTPUT_PATH}")

Reconstruct behavior log-probs for negatives…
Loading behavior model: imge/rf_llama_merged_v2


Loading checkpoint shards: 100%|██████████| 7/7 [00:03<00:00,  1.84it/s]


Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -49.75
seq_prob: 2.481541837659083e-22
log_prob_avg: -12.4375
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -48.25
seq_prob: 1.1117307432712692e-21
log_prob_avg: -12.0625
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -55.0
seq_prob: 1.2989320556496763e-24
log_prob_avg: -13.75
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -48.0
seq_prob: 1.4227506535912076e-21
log_prob_avg: -12.0
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -46.5
seq_prob: 6.3792168840089494e-21
log_prob_avg: -11.625
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -45.25
seq_

log_prob_sum: -47.25
seq_prob: 3.017554874593445e-21
log_prob_avg: -11.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -48.75
seq_prob: 6.716706573930585e-22
log_prob_avg: -12.1875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -47.0
seq_prob: 3.864587821847745e-21
log_prob_avg: -11.75
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -48.5
seq_prob: 8.66885281955573e-22
log_prob_avg: -12.125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -44.75
seq_prob: 3.6845933205562065e-20
log_prob_avg: -11.1875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -50.0
seq_prob: 1.9273308272485545e-22
log_prob_avg: -12.5
Encode prompt and action
Concatinate prompt and acti

log_prob_sum: -44.75
seq_prob: 3.6845933205562065e-20
log_prob_avg: -11.1875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -44.0
seq_prob: 7.792703114739563e-20
log_prob_avg: -11.0
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -49.25
seq_prob: 4.0862722260119567e-22
log_prob_avg: -12.3125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -48.5
seq_prob: 8.66885281955573e-22
log_prob_avg: -12.125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -43.25
seq_prob: 1.6432439176733427e-19
log_prob_avg: -10.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -44.25
seq_prob: 6.056285572868247e-20
log_prob_avg: -11.0625
Encode prompt and action
Concatinate prompt and 

log_prob_sum: -42.0
seq_prob: 5.759824041329242e-19
log_prob_avg: -10.5
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -47.25
seq_prob: 3.017554874593445e-21
log_prob_avg: -11.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -49.25
seq_prob: 4.0862722260119567e-22
log_prob_avg: -12.3125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -49.0
seq_prob: 5.227781471335135e-22
log_prob_avg: -12.25
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -51.25
seq_prob: 5.542110104105285e-23
log_prob_avg: -12.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -42.75
seq_prob: 2.710505431213761e-19
log_prob_avg: -10.6875
Encode prompt and action
Concatinate prompt and ac

log_prob_sum: -48.0
seq_prob: 1.4227506535912076e-21
log_prob_avg: -12.0
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -53.5
seq_prob: 5.816113682013476e-24
log_prob_avg: -13.375
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -50.5
seq_prob: 1.166324663699769e-22
log_prob_avg: -12.625
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -52.5
seq_prob: 1.5819829215076654e-23
log_prob_avg: -13.125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -50.5
seq_prob: 1.166324663699769e-22
log_prob_avg: -12.625
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -47.75
seq_prob: 1.826414792517085e-21
log_prob_avg: -11.9375
Encode prompt and action
Concatinate prompt and action

log_prob_sum: -520.0
seq_prob: 0.0
log_prob_avg: -12.6875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -366.0
seq_prob: 0.0
log_prob_avg: -12.1875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -528.0
seq_prob: 0.0
log_prob_avg: -11.75
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -520.0
seq_prob: 0.0
log_prob_avg: -12.6875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -528.0
seq_prob: 0.0
log_prob_avg: -11.75
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -338.0
seq_prob: 0.0
log_prob_avg: -12.5
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -500.0
seq_prob: 0.0
log_prob_avg: -11.625
En

log_prob_sum: -226.0
seq_prob: 0.0
log_prob_avg: -11.875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -280.0
seq_prob: 0.0
log_prob_avg: -11.1875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -226.0
seq_prob: 0.0
log_prob_avg: -11.875
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -236.0
seq_prob: 0.0
log_prob_avg: -11.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -236.0
seq_prob: 0.0
log_prob_avg: -11.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -236.0
seq_prob: 0.0
log_prob_avg: -11.8125
Encode prompt and action
Concatinate prompt and action
Remove overflow, what is longer than max_len
log_prob_sum: -304.0
seq_prob: 0.0
log_prob_avg: -12.1

In [6]:
# Path to the results directory
DATA_PATH = "./referencegame_data_v2.json"
OUTPUT_PATH = "./referencegame_data_DS_v2.json"

In [7]:
# -----------------------------------------
# 2) Data Sharing
# -----------------------------------------

def data_sharing_positive(data_set, index, data_row: dict) -> dict:
    if data_row['reward'] == 1:
        
        if data_row['role'] == 'gen':
            return{
                "role": 'comp',
                "id": len(data_set) + index + 1,
                "source": data_row["source"] + "_DS", # indication that the source came via Data Sharing
                "model_version": data_row["behavior_model_id"],
                "prompt_speaker": data_row["prompt_speaker"],
                "utterance": data_row["utterance"],
                "prompt_listener": data_row["prompt_listener"],
                "guess": data_row["guess"],
                "target_text": data_row["target_text"],
                "status": data_row["status"],
                "reward": data_row["reward"],
            }
        
        elif data_row['role'] == 'comp':
            return{
                "role": 'gen',
                "id": len(data_set) + index + 1,
                "source": data_row["source"] + "_DS", # indication that the source came via Data Sharing
                "model_version": data_row["behavior_model_id"],
                "prompt_speaker": data_row["prompt_speaker"],
                "utterance": data_row["utterance"],
                "prompt_listener": data_row["prompt_listener"],
                "guess": data_row["guess"],
                "target_text": data_row["target_text"],
                "status": data_row["status"],
                "reward": data_row["reward"],
            }

def expand_cross_task(data_rows: List[dict]) -> List[dict]:
    print('Conduct Data Sharing!')
    new_data = [data_sharing_positive(data_rows, index, r) for index, r in enumerate(data_rows) if r["reward"] == 1]
#     print(new_data)
    data_rows.extend(new_data)
    return data_rows

In [8]:
with open(DATA_PATH, 'r') as f:
    data = json.load(f)
    
data_ds = expand_cross_task(data)
print(len(data_ds))

Conduct Data Sharing!
677


In [9]:
for index, dp in enumerate(data_ds):
    dp['id'] = index + 1
    

In [10]:
with open(OUTPUT_PATH, 'w') as f:
    json.dump(data_ds, f, indent=4)